## Order of Parameters:

$\alpha_{P,0}$, $\alpha_{P,2}$, $\alpha_{B,0}$, $\alpha_{B,P}$, $b_1$, $b_2/2$, $b_{K^2}$, $b_{td}$, $b_{\nabla^2}$, $b_3/6$

### List of the Needed Plots:

One full EFT with the td and laplacian bias vs reduced EFT.

New halo sample vs old halo sample.

New redshift, show original and linear evolved vs z=8.

One with the linear LIMD model.

In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
from getdist import plots, MCSamples
from IPython.display import display, Latex

import matplotlib
matplotlib.style.use('boone')

%matplotlib inline

cmap = plt.get_cmap("tab10")
colors = [cmap(i) for i in np.arange(10)]

g = plots.getSubplotPlotter(width_inch=7)
g.settings.axes_fontsize = 16     # tick number font size
g.settings.axes_labelsize = 22    # axis label font size (Om, sigma8)
g.settings.legend_fontsize = 18   # legend text
g.settings.linewidth = 2.0        # (optional) thicker contour lines
g.settings.figure_legend_loc = 'upper right'

In [2]:
z8_dir = 'Fits_z8/'
z5_dir = 'Fits_z5/'
z8_7e10_dir = '7E10_Fits_z8/'
amp_dir = 'Amplitude/'
lam_dir = 'Lambda/'

b1_ind = 4
b2_ind = 5
bK2_ind = 6
btd_ind = 7
blap_ind = 8
b3_ind = 9

extra_factors = np.array([1,1,1,1,1,2,1,1,1,6])
names = np.array([r'$\alpha_{P,0}$', r'$\alpha_{P,2}$', r'$\alpha_{B,0}$', r'$\alpha_{B,P}$', 
                  'b_1', 'b_2', 'b_{K^2}', 'b_{td}', r'$b_{\nabla^2}$', 'b_3'])

In [3]:
def get_fast_chains(params_file, bias, extra_factors=extra_factors, names=names, phases=25):
    '''
    Overarching method for getting the chains for plotting.

    params_file will be the path to the file with all of the parameter best fit values.
    extra_factors accounts for my b2 and b3 being too small by factors of 2 and 6 respectively.
    names is a list of the names and labels for the chain.
    bias is a boolean numpy array. Only the 'True' entries will be included in the chain.
    '''
    fits = np.load(params_file)
    
    fit_params = []
    for ph in np.arange(phases):
        fit_params.append(fits[ph]*extra_factors)
    fit_params = np.array(fit_params)

    C_params = np.cov(fit_params,rowvar=False)

    mean = np.average(fit_params, axis=0)
    
    x = np.random.multivariate_normal(np.array(mean),C_params,30000)

    b = []
    
    for i in np.arange(len(bias)):
        if bias[i]:
            b.append(np.array(x[:,i]).astype(np.float64))

    ssa = np.array(b).T

    chain = MCSamples(samples=ssa,weights=np.ones(30000), names = names[bias], labels = names[bias])

    return fit_params[:,bias], chain

In [4]:
def format_mean_sem(array, sigfigs=2):
    """
    Return 'mean ± standard error of the mean', with the SEM shown
    to `sigfigs` significant figures.
    """
    array = np.asarray(array)
    mean = np.mean(array)
    sem = np.std(array, ddof=1) / np.sqrt(array.size)
    decimals = sigfigs - 1 - math.floor(math.log10(abs(sem)))
    decimals = max(0, decimals)
    return f"{mean:.{decimals}f} ± {sem:.{decimals}f}"

def b2(b1):
    return 0.412 - 2.143*b1 + 0.929*b1**2 + 0.008*b1**3

def b3(b1):
    return -1.028 + 7.646*b1 - 6.227*b1**2 + 0.912*b1**3

def bK2(b1):
    return -2/7*(b1-1)

def btd(b1):
    return 23/42*(b1-1)

In [5]:
def print_results(params, bias_calculated):
    '''Print the results as well as the predictions'''
    b1 = np.average(params[:,0])
    display(Latex(f'${names[b1_ind]}$: {format_mean_sem(params[:,0])}'))

    next_ind = 1
    if bias_calculated[b2_ind]:
        display(Latex(f'${names[b2_ind]}$: {format_mean_sem(params[:,next_ind])}, Predicted: {round(b2(b1),1)}'))
        next_ind += 1

    if bias_calculated[bK2_ind]:
        display(Latex(f'${names[bK2_ind]}$: {format_mean_sem(params[:,next_ind])}, Predicted: {round(bK2(b1),1)}'))
        next_ind += 1

    if bias_calculated[btd_ind]:
        display(Latex(f'${names[btd_ind]}$: {format_mean_sem(params[:,next_ind])}, Predicted: {round(btd(b1),1)}'))
        next_ind += 1

    if bias_calculated[blap_ind]:
        display(Latex(f'{names[blap_ind]}: {format_mean_sem(params[:,next_ind])}'))
        next_ind += 1

    if bias_calculated[b3_ind]:
        display(Latex(f'${names[b3_ind]}$: {format_mean_sem(params[:,next_ind])}, Predicted: {round(b3(b1),1)}'))
    
    return None

In [6]:
full_bias=[False,False,False,False,True,True,True,False,False,False]

full_params, full_chain = get_fast_chains(z8_dir + 'Reduced_Best_Fits.npy', full_bias)
print_results(full_params, full_bias)

full_amp, full_amp_chain = get_fast_chains(amp_dir + 'Reduced_Best_Fits.npy', full_bias)
#print_results(full_amp, full_bias)

full_lam, full_lam_chain = get_fast_chains(lam_dir + 'Reduced_Best_Fits.npy', full_bias)
#print_results(full_lam, full_bias)

full_k, full_k_chain = get_fast_chains('Shift_kmax/Reduced_Best_Fits.npy', full_bias)
print_results(full_k, full_bias)

Removed no burn in


<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

Removed no burn in
Removed no burn in
Removed no burn in


<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

In [7]:
full_v1, full_v1_chain = get_fast_chains('New_Cov/v1_Reduced_Best_Fits.npy', full_bias)
# print_results(full_v1, full_bias)

full_v2, full_v2_chain = get_fast_chains('New_Cov/v2_Reduced_Best_Fits.npy', full_bias)
# print_results(full_v2, full_bias)

full_cross, full_cross_chain = get_fast_chains('New_Cov_w_Cross/v2_Reduced_Best_Fits.npy', full_bias)
print_results(full_cross, full_bias)

full_no_bin, full_no_bin_chain = get_fast_chains('New_Cov_w_Cross_no_Bins/v2_Reduced_Best_Fits.npy', full_bias)
print_results(full_no_bin, full_bias)

Removed no burn in
Removed no burn in
Removed no burn in


<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

Removed no burn in


<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

In [8]:
# full_7E10_params, full_7E10_chain = get_fast_chains(z8_7e10_dir + 'Reduced_Best_Fits.npy', full_bias)
# print_results(full_7E10_params, full_bias)

In [9]:
# orig_5_params, orig_5_chain = get_fast_chains(z5_dir + 'Reduced_Best_Fits.npy', full_bias)
# print_results(orig_5_params, full_bias)

In [10]:
# z5_boost = ((np.average(orig_5_params[:,0])*1.5)-0.5) / np.average(orig_5_params[:,0])
# z5_factors = np.array([1,1,1,1,1*z5_boost,2*z5_boost**2,1*z5_boost**2,1,1,6])

# full_5_params, full_5_chain = get_fast_chains(z5_dir + 'Reduced_Best_Fits.npy', full_bias, extra_factors=z5_factors)
# print('Ignore predicted for this, this has been rescaled.')
# print_results(full_5_params, full_bias)

In [11]:
# eft_bias=[False,False,False,False,True,True,True,True,True,False]

# eft_params, eft_chain = get_fast_chains(z8_dir + 'EFT_Best_Fits.npy', eft_bias)
# print_results(eft_params, eft_bias)

In [12]:
lin_bias=[False,False,False,False,True,True,False,False,False,True]

lin_params, lin_chain = get_fast_chains(z8_dir + 'Lin_b3_Best_Fits.npy', lin_bias)
print_results(lin_params, lin_bias)

lin_amp, lin_amp_chain = get_fast_chains(amp_dir + 'Lin_b3_Best_Fits.npy', lin_bias)
print_results(lin_amp, lin_bias)

lin_lam, lin_lam_chain = get_fast_chains(lam_dir + 'Lin_b3_Best_Fits.npy', lin_bias)
print_results(lin_lam, lin_bias)

lin_k, lin_k_chain = get_fast_chains('Shift_kmax/Lin_b3_Best_Fits.npy', lin_bias)
print_results(lin_k, lin_bias)

Removed no burn in


<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

Removed no burn in


<IPython.core.display.Latex object>

... (6 outputs left out)

Removed no burn in


<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

In [13]:
# lin_7E10_params, lin_7E10_chain = get_fast_chains(z8_7e10_dir + 'Lin_b3_Best_Fits.npy', lin_bias)
# print_results(lin_7E10_params, lin_bias)

# lin_5_params, lin_5_chain = get_fast_chains(z5_dir + 'Lin_b3_Best_Fits.npy', lin_bias)
# print_results(lin_5_params, lin_bias)

In [14]:
# g.triangle_plot([eft_chain] + [full_chain],
#                 names[eft_bias], 
#                 colors=colors, contour_colors=colors,
#                 legend_labels=['Full EFT', 'Reduced EFT, Baseline Model'],
#                 filled=True, contour_lws=1.2, contour_ls = '-')
# g.fig.suptitle(r"Full vs. Reduced EFT, $z=8$", y = 1.02, fontsize=20)
# g.fig.savefig('v2_Plots/eft.pdf', format='pdf', bbox_inches='tight',  # include everything that sticks out
#               pad_inches=0.02)
# plt.close()
# # plt.show()

In [15]:
# g.triangle_plot([full_chain] + [full_7E10_chain],
#                 names[full_bias], 
#                 colors=colors, contour_colors=colors,
#                 legend_labels=[r'$1\times 10^{11}$ $h^{-1}M_\odot$', r'$7\times 10^{10}$ $h^{-1}M_\odot$'],
#                 filled=True, contour_lws=1.2, contour_ls = '-')
# g.fig.suptitle(r"Mass Threshold Dependence, $z=8$", y = 1.02, fontsize=20)
# g.fig.savefig('v2_Plots/mass.pdf', format='pdf', bbox_inches='tight',  # include everything that sticks out
#               pad_inches=0.02)
# plt.close()
# # plt.show()

In [16]:
# g.triangle_plot([full_chain] + [orig_5_chain] + [full_5_chain],
#                 names[full_bias], 
#                 colors=colors, contour_colors=colors,
#                 legend_labels=[r'$z=8$', r'$z=5$', r'$z=5$, Linearly Evolved'],
#                 filled=True, contour_lws=1.2, contour_ls = '-')
# g.fig.suptitle(r"Redshift Evolution, $z=5$ to $z=8$", y = 1.02, fontsize=20)
# g.fig.savefig('v2_Plots/redshift.pdf', format='pdf', bbox_inches='tight',  # include everything that sticks out
#               pad_inches=0.02)
# plt.close()
# # plt.show()

In [17]:
# g.triangle_plot([lin_chain],
#                 names[lin_bias], 
#                 colors=colors, contour_colors=colors,
#                 legend_labels=[],
#                 filled=True, contour_lws=1.2, contour_ls = '-')
# g.fig.suptitle(r"Linear LIMD Model, $z=8$", y = 1.02, fontsize=20)
# g.fig.savefig('v2_Plots/linear.pdf', format='pdf', bbox_inches='tight',  # include everything that sticks out
#               pad_inches=0.02)
# plt.close()
# # plt.show()

In [18]:
g.triangle_plot([full_chain] + [full_amp_chain] + [full_lam_chain],
                names[full_bias], 
                colors=colors, contour_colors=colors,
                legend_labels=[r'Baseline', r'Random Amplitudes', r"$\Lambda' \approx 1.2 h$ Mpc$^{-1}$"],
                filled=True, contour_lws=1.2, contour_ls = '-')
g.fig.suptitle(r"Template Construction Dependence", y = 1.02, fontsize=20)
g.fig.savefig('v2_Plots/template.pdf', format='pdf', bbox_inches='tight',  # include everything that sticks out
              pad_inches=0.02)
plt.close()
# plt.show()

In [19]:
g.triangle_plot([full_chain] + [full_k_chain] + [full_chain],
                names[full_bias], 
                colors=[cmap(0), cmap(1), cmap(0)], contour_colors=[cmap(0), cmap(1), cmap(0)],
                legend_labels=[r'$k_{max}\approx 0.4h$ Mpc$^{-1}$ (Baseline)', r'$k_{max}\approx 0.2h$ Mpc$^{-1}$'],
                filled=True, contour_lws=1.2, contour_ls = '-')
g.fig.suptitle(r"Fitting $k_{max}$ Dependence", y = 1.02, fontsize=20)
g.fig.savefig('v2_Plots/kmax.pdf', format='pdf', bbox_inches='tight',  # include everything that sticks out
              pad_inches=0.02)
plt.close()
# plt.show()

In [20]:
g.triangle_plot([full_chain] + [full_v1_chain] + [full_v2_chain],
                names[full_bias], 
                colors=colors, contour_colors=colors,
                legend_labels=[r'Sample Covariance (Baseline)', r'Analytic, No Shot Noise', r'Analytic, With Shot Noise'],
                filled=True, contour_lws=1.2, contour_ls = '-')
g.fig.suptitle(r"Covariance Dependence", y = 1.02, fontsize=20)
g.fig.savefig('v2_Plots/cov.pdf', format='pdf', bbox_inches='tight',  # include everything that sticks out
              pad_inches=0.02)
plt.close()
# plt.show()

In [21]:
g.triangle_plot([full_chain] + [full_cross_chain] + [full_no_bin_chain],
                names[full_bias], 
                colors=colors, contour_colors=colors,
                legend_labels=[r'Sample Covariance (Baseline)', r'Analytic + Cross', r'Analytic + Cross, Unbinned $P$'],
                filled=True, contour_lws=1.2, contour_ls = '-')
g.fig.suptitle(r"Covariance Extensions", y = 1.02, fontsize=20)
g.fig.savefig('v2_Plots/cov_ext.pdf', format='pdf', bbox_inches='tight',  # include everything that sticks out
              pad_inches=0.02)
plt.close()
# plt.show()